In [ ]:
from pathlib import Path
import os, random, json, csv, logging
from typing import Dict, List
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
import torchvision.transforms as T

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix, classification_report

# ============================================================
# CHOOSE TASK HERE
# ============================================================
TASK = "spatial"  # "spatial" or "threat"

# spatial = isolated vs overlap
# threat  = non_contraband vs contraband

# ============================================================
# CONFIG
# ============================================================
class Config:
    PROCESSED_ROOT = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2_two_models/gray")
    INDEX_CSV = PROCESSED_ROOT / "index.csv"

    IMAGE_MODE = "gray"
    IMAGE_SIZE = 1024

    BATCH_SIZE = 8
    EPOCHS = 50
    NUM_WORKERS = 2

    LR_HEAD = 5e-4
    LR_FULL = 2e-5
    WEIGHT_DECAY = 1e-3
    UNFREEZE_EPOCH = 1

    USE_AMP = True
    EARLY_STOPPING_PATIENCE = 10

    SEED = 42
    GRAY_MEAN = (0.5,)
    GRAY_STD = (0.25,)

    if TASK == "spatial":
        LABEL_DIR = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2_two_models/gray/spatial_overlap_isolated")
        CLASS_NAMES = ["isolated", "overlap"]  # 0, 1
        MODEL_DIR = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2_two_models/spatial_overlap_isolated")
    elif TASK == "threat":
        LABEL_DIR = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2_two_models/gray/threat_contraband_noncontraband")
        CLASS_NAMES = ["non_contraband", "contraband"]  # 0, 1
        MODEL_DIR = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2_two_models/threat_contraband_noncontraband")
    else:
        raise ValueError("TASK must be 'spatial' or 'threat'")

    TRAIN_LABELS_JSON = LABEL_DIR / "train.json"
    VAL_LABELS_JSON = LABEL_DIR / "val.json"

    MODEL_OUT_PATH = MODEL_DIR / "model.pt"
    CKPT_DIR = MODEL_DIR / "checkpoints"

    @classmethod
    def create_dirs(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        cls.CKPT_DIR.mkdir(parents=True, exist_ok=True)

config = Config()
config.create_dirs()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(config.MODEL_DIR / f"training_{TASK}.log"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"TASK={TASK}")
logger.info(f"Using device: {device}")
logger.info(f"Classes: {config.CLASS_NAMES}")

# ============================================================
# SEED
# ============================================================
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(config.SEED)

def seed_worker(worker_id):
    worker_seed = config.SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(config.SEED)

# ============================================================
# DATA
# ============================================================
def read_index_csv(index_csv_path: Path) -> List[Dict]:
    rows = []
    with open(index_csv_path, "r", newline="") as f:
        rows.extend(csv.DictReader(f))
    return rows

def load_label_map(label_path: Path) -> Dict[str, int]:
    data = json.load(open(label_path, "r"))
    out = {}

    for item in data:
        filepath = item["image"].replace("\\", "/").strip()
        fname = Path(filepath).name
        y = int(item["class_id"])
        out[filepath] = y
        out[fname] = y

    return out

index_rows = read_index_csv(config.INDEX_CSV)
train_label_map = load_label_map(config.TRAIN_LABELS_JSON)
val_label_map = load_label_map(config.VAL_LABELS_JSON)

class SingleTaskDataset(Dataset):
    def __init__(self, index_rows, processed_root, split, label_map, transform=None):
        self.processed_root = processed_root
        self.split = split
        self.transform = transform
        self.filepaths = []
        self.labels = []

        missing = []
        for r in index_rows:
            fp = r["filepath"].replace("\\", "/").strip()
            if r["split"].lower() != split:
                continue

            fname = Path(fp).name
            label = label_map.get(fp, label_map.get(fname))

            # Skip samples that are not part of this task's label json
            if label is None:
                continue

            img_path = processed_root / fp
            if not img_path.exists():
                missing.append(str(img_path))
                continue

            self.filepaths.append(fp)
            self.labels.append(int(label))

        if missing:
            raise FileNotFoundError(f"Missing image files: {missing[:10]}")

        logger.info(f"{TASK} {split}: {len(self.filepaths)} samples")
        logger.info(f"{TASK} {split} distribution: {Counter(self.labels)}")

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.processed_root / self.filepaths[idx]
        img = Image.open(img_path).convert("L" if config.IMAGE_MODE == "gray" else "RGB")

        if self.transform:
            img = self.transform(img)

        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, y

train_transform = T.Compose([
    T.RandomResizedCrop(config.IMAGE_SIZE, scale=(0.95, 1.0), ratio=(0.98, 1.02)),
    T.RandomAffine(degrees=2, translate=(0.01, 0.01), scale=(0.99, 1.01)),
    T.ToTensor(),
    T.Normalize(config.GRAY_MEAN, config.GRAY_STD),
])

val_transform = T.Compose([
    T.Resize(int(config.IMAGE_SIZE * 1.10)),
    T.CenterCrop(config.IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(config.GRAY_MEAN, config.GRAY_STD),
])

train_ds = SingleTaskDataset(index_rows, config.PROCESSED_ROOT, "train", train_label_map, train_transform)
val_ds = SingleTaskDataset(index_rows, config.PROCESSED_ROOT, "val", val_label_map, val_transform)

# Weighted sampler gives roughly balanced mini-batches
class_counts = np.bincount(train_ds.labels, minlength=2)
class_counts = np.maximum(class_counts, 1)
sample_weights = [1.0 / class_counts[y] for y in train_ds.labels]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

def make_class_weights(labels):
    counts = np.bincount(labels, minlength=2)
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (2 * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 0.8, 1.5)
    return torch.tensor(weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=make_class_weights(train_ds.labels), label_smoothing=0.03)

train_loader = DataLoader(
    train_ds,
    batch_size=config.BATCH_SIZE,
    sampler=sampler,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

# ============================================================
# MODEL
# ============================================================
class SimpleCNN_Binary(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

in_channels = 1 if config.IMAGE_MODE == "gray" else 3
model = SimpleCNN_Binary(in_channels=in_channels, num_classes=2).to(device)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

for p in model.features.parameters():
    p.requires_grad = False

optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=config.LR_HEAD, weight_decay=config.WEIGHT_DECAY)
scheduler = None
scaler = GradScaler("cuda") if config.USE_AMP and device.type == "cuda" else None

logger.info(f"Trainable parameters head only: {count_parameters(model):,}")

# ============================================================
# TRAIN / VAL
# ============================================================
def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    all_labels = []
    all_preds = []
    all_probs = []

    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        for imgs, y in loader:
            imgs = imgs.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            if config.USE_AMP and scaler is not None and train:
                with autocast("cuda"):
                    logits = model(imgs)
                    loss = criterion(logits, y)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(imgs)
                loss = criterion(logits, y)

                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            bs = imgs.size(0)
            total_loss += loss.item() * bs
            correct += (preds == y).sum().item()
            total += bs

            all_labels.extend(y.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs[:, 1].detach().cpu().numpy().tolist())

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0

    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auc": float(auc),
        "cm": cm.tolist(),
    }

best_val_loss = float("inf")
epochs_no_improve = 0

logger.info("=" * 80)
logger.info(f"Training {TASK} model")
logger.info(f"0={config.CLASS_NAMES[0]}, 1={config.CLASS_NAMES[1]}")
logger.info("=" * 80)

for epoch in range(1, config.EPOCHS + 1):
    if epoch == config.UNFREEZE_EPOCH:
        logger.info("Unfreezing full model")
        for p in model.parameters():
            p.requires_grad = True

        optimizer = optim.AdamW(model.parameters(), lr=config.LR_FULL, weight_decay=config.WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            patience=4,
            factor=0.7,
            min_lr=1e-7,
        )

    train_m = run_epoch(model, train_loader, train=True)
    val_m = run_epoch(model, val_loader, train=False)

    if scheduler is not None:
        scheduler.step(val_m["loss"])

    if val_m["loss"] < best_val_loss:
        best_val_loss = val_m["loss"]
        epochs_no_improve = 0

        torch.save(model.state_dict(), config.CKPT_DIR / "best.pt")
        torch.save({
            "epoch": epoch,
            "task": TASK,
            "class_names": config.CLASS_NAMES,
            "model_state": model.state_dict(),
            "train_metrics": train_m,
            "val_metrics": val_m,
        }, config.CKPT_DIR / "best_checkpoint.pt")

        logger.info("✓ Saved new best model")
    else:
        epochs_no_improve += 1

    logger.info("")
    logger.info(f"Epoch {epoch}/{config.EPOCHS}")
    logger.info(f"Train: loss={train_m['loss']:.4f}, acc={train_m['acc']:.3f}, f1={train_m['f1']:.3f}, auc={train_m['auc']:.3f}, cm={train_m['cm']}")
    logger.info(f"Val:   loss={val_m['loss']:.4f}, acc={val_m['acc']:.3f}, f1={val_m['f1']:.3f}, auc={val_m['auc']:.3f}, cm={val_m['cm']}")

    if epochs_no_improve >= config.EARLY_STOPPING_PATIENCE:
        logger.info(f"Early stopping at epoch {epoch}")
        break

torch.save(model.state_dict(), config.MODEL_OUT_PATH)

with open(config.MODEL_DIR / f"metrics_{TASK}.json", "w") as f:
    json.dump({
        "task": TASK,
        "class_names": config.CLASS_NAMES,
        "best_val_loss": best_val_loss,
    }, f, indent=2)

logger.info(f"Saved final model to: {config.MODEL_OUT_PATH}")
logger.info("Done")
